# Hallucination Management & Grounding

**Use case:** Compare ungrounded and grounded generation.

This notebook demonstrates the Day 4 Responsible AI / Governance concept in a simple end-to-end flow.

**Total steps:** 10

## Installation

```bash
pip install pandas numpy matplotlib langchain-openai python-dotenv
```

## LLM setup
Create a `.env` file:

```text
OPENAI_API_KEY=your_key_here
```

This notebook uses `langchain_openai.ChatOpenAI`.

## Step 1 - Define a trusted knowledge source

In [ ]:
knowledge = {'refund_window':'30 days','warranty':'12 months','support_email':'support@example.com','store_hours':'9 AM to 8 PM'}
print(knowledge)


## Step 2 - Define a user question

In [ ]:
question = 'What is the refund window and warranty period?'


## Step 3 - Initialize the LLM

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import pandas as pd
load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini',temperature=0)


## Step 4 - Ask without grounding

In [ ]:
ungrounded = llm.invoke(question).content
print(ungrounded)


## Step 5 - Build a grounded prompt

In [ ]:
grounded_prompt = f'''Answer only from this trusted knowledge: {knowledge}. If the answer is not present, say I do not know. Question: {question}'''


## Step 6 - Generate grounded answer

In [ ]:
grounded = llm.invoke(grounded_prompt).content
print(grounded)


## Step 7 - Check required facts

In [ ]:
facts = ['30 days','12 months']
checks = {fact:fact.lower() in grounded.lower() for fact in facts}
print(checks)


## Step 8 - Detect unsupported claims

In [ ]:
unsupported = 'lifetime' in grounded.lower() or '60 days' in grounded.lower()
print('Unsupported claim detected:',unsupported)


## Step 9 - Assign groundedness status

In [ ]:
status = 'PASS' if all(checks.values()) and not unsupported else 'REVIEW'
print('Groundedness status:',status)


## Step 10 - Save evidence

In [ ]:
pd.DataFrame([{'question':question,'answer':grounded,'status':status}]).to_csv('groundedness_evidence.csv',index=False)
print('Saved groundedness_evidence.csv')
